** Important **
For the kernel, please select base (Python 3.9.12)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import glob
import os
sys.path.insert(0,"/home/ws/sk6801/sw/UCSD_analysis/sandpro")
import sandpro
import configparser
import json
import scipy.stats
from matplotlib.colors import LogNorm

from scipy.optimize import curve_fit
import datetime
import pandas as pd
# %run ../WaveformProcessor.py
# %run ../FastProcessing.py
# %run ../FitSPE.py

sys.path.insert(0,"../src/")
import common.d2d as d2d
import common.utils as util
import data_processing.fast_processor_all_channel as fast_processor

### Check which datasets are available

In [ ]:
# path = "/home/daqtest/DAQ/SandyAQ/softlink_to_data/noise_time_evolution_before_threshold_calibration_implemented/"
# path = "/home/daqtest/DAQ/SandyAQ/softlink_to_data/all_data/20240819_T102_50V_5.0sig/"
# path = "/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/20241010_2_T98_47V_5.5sig"

# path = "/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/20241011_T98_45V_4.0sig"
# path = "/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/20241015_2_T98_47V_1.5sig"
# path = "/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/20241016_T98_47V_4.0sig"
# path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202405_202406_GXe_threshold/threshold_runs/1_"
# path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202405_202406_GXe_threshold/rms_runs/8_4sig/"
# path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe/20241018_all_T98_all_voltages_3.0sig/"
# path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe_tritium/20241031_all_1_T98_all_voltages_10.0sig/meta_config_all_20241031_113512.json"
# path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe_tritium/20241030_all_1_T98_all_voltages_5.0sig/meta_config_all_20241030_180916.json"
# path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe_new/20241030_all_1_T98_all_voltages_6.0sig/meta_config_all_20241030_171835.json"
# path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe_tritium/20241031_all_1_T98_all_voltages_6.0sig/meta_config_all_20241031_113258.json",
# path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe_new/20241030_all_1_T98_all_voltages_6.0sig/meta_config_all_20241030_170524.json"
path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202405_202409_GXe/20240902_T106_47V_5.0sig/meta_config_22_2459_20240903_133513.json"
# paths = ["/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/20241014_2_T98_47V_1.5sig"]

#Gas paths
# paths.append("/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/20240912_T102_51V_8.0sig")
# paths.append("/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/20240924_T102_48V_5.5sig")
# paths.append("/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/20240923_T102_49V_6.5sig")
# paths.append("/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/20240920_T102_49V_6.5sig")
# # GXe_path = "/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/gain_run_1/T98_47V/"
# # path = "/home/daqtest/DAQ/SandyAQ/softlink_to_data/all_data/T100_47_5,5sig_V/" # check waveform

plot = True # if want to replot the waveforms
# ignore_channel_list = np.array([2,8,11,14,20,23])
ignore_channel_list = np.array([])
channels = range(0,24) # or a list of channels
# channels = [22] # or a list of channels
 
# ##########################
# Load the meta data files

def find_meta_data_files(path, selected_channel = channels):

    
    
    # Check if the path is a directory
    if os.path.isdir(path):
        meta_data_list = glob.glob(f"{path}/*meta*")

    # Sort the file paths based on the extracted date and time in ascending order (oldest to newest)
    new_meta_data_list = sorted(meta_data_list, key=util.extract_date_meta_data)
    new_meta_data_list = sorted(new_meta_data_list[-24:], key=util.extract_channel_meta_data)
    print(new_meta_data_list)

    ## select channels
    selected_list = []
    for i in selected_channel:
        selected_list.append(new_meta_data_list[i])

    return selected_list

In [ ]:
if not os.path.exists(path):
    print(f"Path {path} does not exist.")
    raise FileNotFoundError(f"Path {path} does not exist.")

# if the path is a file, return it directly
if os.path.isfile(path) and path.endswith('.json'):
    meta_data_files = path
    meta_dir_path = os.path.dirname(meta_data_files)

In [ ]:
FastProcessor = fast_processor.FastProcessor(meta_dir_path, meta_data_files, 
            ignore_channel_list=ignore_channel_list)
FastProcessor.process_runs()

# FastProcessorList.append(FastProcessor)
# run_tags.append(FastProcessor.get_run_tag(meta_data_files[0]))

In [ ]:
# height_V_gas = [np.array([])] * len(channels)
# area_Vns_gas = [np.array([])] * len(channels)
# rise_time_ns_gas = [np.array([])] * len(channels)

# height_V_liquid = [np.array([])] * len(channels)
# area_Vns_liquid = [np.array([])] * len(channels)
# rise_time_ns_liquid = [np.array([])] * len(channels)

# run_tags_cluster = ['GXe/gain_calibration', 'LXe/gain_calibration']

# ### Stack gas events
# for path in paths:
#     meta_data_files = find_meta_data_files(path)
#     FastProcessor = fast_processor.FastProcessor(path, meta_data_files, 
#                 ignore_channel_list=ignore_channel_list)
#     FastProcessor.process_runs()

#     run_tag = FastProcessor.get_run_tag(meta_data_files[0])

#     if run_tag == run_tags_cluster[0]:

#         for channel, fast_info in enumerate(FastProcessor.list_of_fast_info):
#             height_V_gas[channel] = np.concatenate((height_V_gas[channel], fast_info.heights_V))
#             area_Vns_gas[channel] = np.concatenate((area_Vns_gas[channel], fast_info.areas_Vns))
#             rise_time_ns_gas[channel] = np.concatenate((rise_time_ns_gas[channel], fast_info.rise_time_ns))

#     elif run_tag == run_tags_cluster[1]:

#         for channel, fast_info in enumerate(FastProcessor.list_of_fast_info):
#             height_V_liquid[channel] = np.concatenate((height_V_liquid[channel], fast_info.heights_V))
#             area_Vns_liquid[channel] = np.concatenate((area_Vns_liquid[channel], fast_info.areas_Vns))
#             rise_time_ns_liquid[channel] = np.concatenate((rise_time_ns_liquid[channel], fast_info.rise_time_ns))

#     run_tags.append(run_tag)


In [ ]:
plt.close()
FastProcessor.plot_waveforms(
    save_plot = True, 
    show_plot=False, 
    # channels=[0,1,3,4,5,6,7,8,9,10,11,12,13,15,16,17,18,19,20,21,22,23]
)


In [ ]:
# 
plt.close()

fig, axes = plt.subplots(3,8,figsize=(35,15))

# assert(len(FastProcessor.list_of_fast_info) == 24)


for fast_info in FastProcessor.list_of_fast_info:
    height_V = fast_info.heights_V
    area_Vns = fast_info.areas_Vns
    channel = fast_info.channel
    

    plot_row = channel % 3
    plot_col = channel // 3

    # change the rows so that it match with the physical layout of the channels
    if plot_row == 0:
        plot_row = 1
    elif plot_row == 1:
        plot_row = 0

    axes[plot_row,plot_col].hist2d(area_Vns,height_V,
                            bins=[100,100],
                            range=[[-0.1,800],[0,2]],
                            cmap='viridis',
                            norm=LogNorm())
    
    axes[plot_row,plot_col].set_title(f"Channel {fast_info.channel}")

# set the labels
for ax in axes.flat:
    ax.set(xlabel='Area [V*ns]', ylabel='Height [V]')

# Set plot title
plt.suptitle(f"Dataset: {path.split('/')[-1]}")

# Move title upwards
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()

### Peak test

In [ ]:
raw_waveform = FastProcessor.list_of_fast_info[0].EventProcessor.waveform

In [ ]:
waveform = raw_waveform["data_per_channel"][:,5,:]
waveform.shape

In [ ]:

peak_width_sample = 30
smooth_waveform = rolling_window(waveform, window_size, axis=1)

# rmb to change waveform to filtered waveform
# add rolling window to smooth the data, roll every 3 points
# test_averaged = np.convolve(waveform, np.ones(window_size)/window_size, mode='valid')

# recalculated the baseline for the smoothed waveform
baseline, baseline_std = get_baseline_for_all_events(smooth_waveform)
threshold = baseline + 5 * baseline_std

threshold = np.expand_dims(threshold, axis=1) # make it a 2D array with shape (n_events, 1)
threshold = np.pad(threshold, ((0, 0), (0, smooth_waveform.shape[1]-1)), 'maximum')

# applying the threshold to find peaks
mask = smooth_waveform > threshold

# mark the start and end of the peaks
diff = np.diff(mask, axis = 1)
points_event = np.where(diff == 1)[0]
points_sample = np.where(diff == 1)[1]
# start points are the points after the rising edge
points_sample[0::2] = points_sample[0::2]+1 


# pair the start and end points of the peaks
# remove the last point if it's odd
#????? FIXME: this is not working, need to group base on events
for i,row in enumerate(np.unique(points_event)):
    points_sample_row = points_sample[points_event == row]
    points_event_row = points_event[points_event == row]

    # remove the last point if it's odd
    if len(points_sample_row) % 2 != 0:
        points_event_row = points_event_row[:-1]  
        points_sample_row = points_sample_row[:-1]  

    # remove the pair if they are too close to each other
    pairs_sample = points_sample_row.reshape(-1, 2)
    pairs_event = points_event_row.reshape(-1, 2)
    print(i)
    print(pairs_sample)
    peak_width = pairs_sample[:,1] - pairs_sample[:,0]
    pairs_sample = pairs_sample[peak_width >= peak_width_sample]
    pairs_event = pairs_event[peak_width >= peak_width_sample]
    print(pairs_sample)

    # flatten the pairs be better handling
    points_sample_row = pairs_sample.ravel()
    points_event_row = pairs_event.ravel()




In [ ]:
np.where(diff == 1)[0]

In [ ]:
PeakInfo(points_event_row, points_sample_row, np.array([]), np.array([]))

test = np.empty((3, 2), dtype=PeakInfo)

test[0,0] = PeakInfo(points_event_row, points_sample_row, np.array([]), np.array([]))

test[0,0].start_time_array
test[0,0].end_time_array
test[0,0].peak_max_array
test[0,0].area_array


In [ ]:
# test_waveform = waveform["data_per_channel"][2161:2162,channel,:][0]

# window_size = 4
# threshold_sig = 5
# peak_width_sample = 30

# test_averaged = np.convolve(test_waveform, np.ones(window_size)/window_size, mode='valid')

# baseline, baseline_std = get_baseline(test_averaged)
# threshold = baseline + threshold_sig * baseline_std

# mask = test_averaged > threshold
# diff = np.diff(mask)

# consecutive_points = window_size*2
# points = np.where(diff == 1)[0]+1
# points[0::2] = points[0::2]+1

# # # reshape the points into pairs
# if len(points) % 2 != 0:
#     points = points[:-1]  # remove the last point if it's odd
# pairs = points.reshape(-1, 2)

# # remove the pair if they are too close to each other
# for pair in pairs:
#     if pair[1] - pair[0] < peak_width_sample:
#         pairs = np.delete(pairs, np.where((pairs == pair).all(axis=1)), axis=0)

# points = pairs.flatten()

# pairs_ns = pairs * 4 # in ns, assuming the sampling rate is 250 MHz (4 ns per sample)
# start_time_array, end_time_array = pairs_ns[:,0], pairs_ns[:,1]
# start_time_array, end_time_array

# peak_max_array = np.empty(pairs.shape[0], dtype=float)
# area_array = np.empty(pairs.shape[0], dtype=float)

# for i, pair in enumerate(pairs):
#     peak_max_array[i] = np.max(test_waveform[pair[0]:pair[1]])
#     area_array[i] = np.sum(test_waveform[points[0]:points[1]])

# peak_info = PeakInfo(start_time_array, end_time_array, peak_max_array, area_array)


In [ ]:
# # plt.plot(test_waveform, color='blue', label='Raw Waveform')
# plt.plot(mask*0.01+0.3, color='blue', label='Threshold Mask')
# # plt.scatter(np.arange(len(test_averaged)),test_averaged, color='orange', label='Averaged Waveform')
# plt.plot(test_waveform, color='orange', label='Averaged Waveform')
# plt.axhline(threshold, color='red', linestyle='--', label='Threshold')
# plt.scatter(points, test_waveform[points], color='green', label='Threshold Crossing Points', zorder=5)
# # plt.scatter(pairs, test_waveform[pairs], color='purple', label='Signal Peaks', zorder=5)

# # plt.xlim(350,370)
# # plt.ylim(0.2,0.4)

In [ ]:
from copy import deepcopy

In [ ]:
# for channel in np.arange(0,12):

test_waveform = waveform["data_per_channel"][:,0,:] # in mV
# result = v_get_waveform(test_waveform, window_size=4, threshold_sig=5, peak_width_sample=30)
event_time_s = waveform['microseconds'][:]/1e6

list_of_peaks_per_channel = []

for i, row in enumerate(test_waveform):
    (print(f"Processing row {i}"))
    result = get_peaks(row, event_tims_s, window_size=4, threshold_sig=5, peak_width_sample=30)
    # print(result.start_time_array, result.end_time_array, result.peak_max_array, result.area_array)

    tmp = deepcopy(result)
    list_of_peaks_per_channel.append(tmp)
    
    

In [ ]:
list_of_peaks_per_channel = np.array(list_of_peaks_per_channel)
for i, peak_info in enumerate(list_of_peaks_per_channel):
    plt.scatter(peak_info.start_time_array, peak_info.peak_max_array, label=f"Channel {i}", s=1)


In [ ]:
list_of_peaks_per_channel = np.array(list_of_peaks_per_channel)
for i, peak_info in enumerate(list_of_peaks_per_channel):
    plt.scatter(peak_info.peak_max_array, peak_info.peak_area_array, label=f"Channel {i}", s=1)

In [ ]:
from scipy.signal import butter, lfilter, freqz

def butter_lowpass(cutoff, fs, order=5):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return b, a

def butter_lowpass_filter(data, cutoff=1e6, fs=250e6, order=1):
    b, a = butter_lowpass(cutoff, fs, order=order)
    y = lfilter(b, a, data)
    return y


In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(test_waveform)
ax.scatter(pairs, test_waveform[pairs], color='red', label='Threshold crossing points')
ax.axhline(threshold)


##### trying to get the cross talk peak, unsuccessful
# plot second derivative
# another rolling window
# test_more_average = np.convolve(test_waveform, np.ones(1)/1, mode='valid')
# first_derivative = np.diff(test_waveform)
# # test_more_average = np.convolve(first_derivative, np.ones(10)/10, mode='valid')
# test_more_average = butter_lowpass_filter(first_derivative, cutoff=1e8, fs=250E6, order=1)
# test_more_average = np.append(np.zeros(10), test_more_average)  # pad with zeros to match the length
# test_more_average = np.append(test_more_average, np.zeros(10)) 

# ax.plot(test_more_average+0.3, color='green', label='First Derivative')
# # second_derivative = np.diff(test_more_average)

# mask_d = test_more_average > 0.001

#  # pad with zeros to match the length
# ax.plot(mask_d*0.01+0.3, color='orange', label='Second Derivative')

In [ ]:
print(np.sum(test_waveform[points[0]:points[1]]))
print(np.sum(test_averaged[points[0]:points[1]]))


In [ ]:
def get_area(test_waveform,sum_window=(0.4,0.6)):
    """
    Return the area of the waveform in the sum window
    Unit: V * ns
    """
    sum_start = int(1000 * sum_window[0])
    sum_end = int(1000 * sum_window[1])

    print(f"Sum window: {sum_start} to {sum_end} samples")

    areas_Vsamples = np.sum(test_waveform[sum_start:sum_end])
    areas_Vns = 4 * areas_Vsamples # now it becomes V * ns (for V1725, 1 sample = 4 ns)

    return areas_Vns

In [ ]:
get_area(test_waveform)

### Noise

In [ ]:
baseline_list = []
for fast_processor in FastProcessor.list_of_fast_info:
    baseline_list.append(fast_processor.baseline_std_V)

channel_list = np.arange(len(baseline_list))

plt.figure(figsize=(10, 6))
plt.bar(channel_list, baseline_list, color='blue', alpha=0.7)
plt.xlabel('Channel')
plt.ylabel('Baseline STD (V)')
plt.title('Baseline STD for Each Channel')
plt.xticks(channel_list, channel_list)
plt.grid(axis='y')
plt.tight_layout()